In [ ]:
# Public-release setup: run from any working directory.
from pathlib import Path
import sys

def find_release_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "docs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the public release directory.")

PROJECT_ROOT = find_release_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PROJECT_ROOT / "results"  # User-supplied artifacts; not included in the release.
FIGURES_ROOT = PROJECT_ROOT / "figures"


# One-Step CH Scatter Plots

This notebook draws paper-style scatter plots comparing:

- actual one-step log-probability change;
- the exact first-order gradient inner product;
- the forward-computable `CH1+CH2` approximation.

It only reads saved CSV outputs. It does not load models or rerun experiments.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root whether this notebook is run from repo root or notebook_for_paper/.
PROJECT_ROOT = find_release_root()
RESULTS_ROOT = PROJECT_ROOT / "results"
FIGURE_DIR = PROJECT_ROOT / "figures" / "paper_one_step_ch_scatters"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"FIGURE_DIR   = {FIGURE_DIR}")

In [ ]:
# Run the data-loader cell below first. Then use `search_runs(...)` or `# Run the data-loader cell below before calling list_available_runs().` to inspect candidates.
# Run the data-loader cell below before calling list_available_runs().

## User Settings

In [ ]:
# ------------- Some setting examples

# SEARCH_KIND = "step1plus"
# RESULT_FAMILY = "step1plus_sft_section3_qwen35_08b"
# DATASET_COMBO = "gsm8k_mmlu"
# MODEL = "Qwen3.5-0.8B"
# STEP1PLUS_CHECKPOINT = "final"  # "base", "mid", "final", or None

# For the new Qwen2.5 jobs, once finished:

# RESULT_FAMILY = "step1plus_sft_section3_qwen25_15b"
# MODEL = "Qwen2.5-1.5B"
# STEP1PLUS_CHECKPOINT = "final"

# or:

# RESULT_FAMILY = "step1plus_sft_section3_qwen25_7b"
# MODEL = "Qwen2.5-7B"
# STEP1PLUS_CHECKPOINT = "final"

# SEARCH_KIND = "step1plus"
# RESULT_FAMILY = "step1plus_sft_section3_qwen35_08b"  # exact family or fuzzy substring; set None to search all results
# DATASET_COMBO = "gsm8k_mmlu"
# MODEL = "Qwen3.5-0.8B"

In [ ]:
# -------------------------
# Data selection
# -------------------------
# SEARCH_KIND can be "one_step", "section3", "step1plus", or None for all.
# DATASET_COMBO accepts either "gsm8k_to_mmlu" or shorthand "gsm8k_mmlu".
SEARCH_KIND = "step1plus"
RESULT_FAMILY = "step1plus_sft_section3_qwen35_08b"  # exact family or fuzzy substring; set None to search all results
DATASET_COMBO = "gsm8k_mmlu"
MODEL = "Qwen3.5-0.8B"

# For Step1+ runs, choose one checkpoint: "base", "mid", "final", or None to use the combined CSV.
STEP1PLUS_CHECKPOINT = "final"

# Optional direct override. Leave as None for automatic lookup by RESULT_FAMILY/DATASET_COMBO/MODEL/SEARCH_KIND.
CSV_PATH_OVERRIDE = None

# If multiple candidates match, choose the most recently modified file by default.
# Set False to display candidates and raise instead.
AUTO_PICK_NEWEST_CANDIDATE = True

# If True, remove rows where actual/first-order/approx values are all finite but actual delta is exactly zero.
DROP_ZERO_ACTUAL = True
EPS = 1e-12

# -------------------------
# Plot controls
# -------------------------
SHOW_FIT_LINE = True
SHOW_CI = True
CI_LEVEL = 95
N_BOOT = 500
RANDOM_SEED = 0
SCATTER_ALPHA = 0.28
SCATTER_SIZE = 12
MARKER = "+"

SAVE_PDF = True
SAVE_PNG = True

# -------------------------
# Labels: edit here without touching plotting code.
# -------------------------
LABELS = {
    "actual": r"$\Delta\log_\pi = \log \pi_{\theta_{t+1}} - \log \pi_{\theta_{t}}$",
    "first_order": r"$lr \cdot \langle \nabla \log \pi_o, \nabla \log \pi_u \rangle$",
    "approx": r"$lr \cdot (CH1 + CH2)$",
    "abs_first_order": r"$|lr \cdot \langle \nabla \log \pi_o, \nabla \log \pi_u \rangle|$",
    "abs_actual": r"$|\Delta \log \pi|$",
    "abs_approx": r"$|lr \cdot (CH1 + CH2)|$",
    "rank_abs_first_order": r"rank($|lr \cdot \langle \nabla \log \pi_o, \nabla \log \pi_u \rangle|$)",
    "rank_abs_approx": r"rank($|lr \cdot (CH1 + CH2)|$)",
    "rank_first_order": r"rank($lr \cdot \langle \nabla \log \pi_o, \nabla \log \pi_u \rangle$)",
    "rank_approx": r"rank($lr \cdot (CH1 + CH2)$)",
}

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLORS = {
    "scatter": "#2f6f9f",
    "fit": "#c75146",
    "reference": "#333333",
    "ci": "#c75146",
}


## Data Loader

In [ ]:
def canonical_text(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def canonical_combo(value):
    text = str(value).lower().replace("_to_", "_").replace("-to-", "_").replace(" to ", "_")
    return canonical_text(text)


def read_json_if_exists(path):
    try:
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None
    return None


def find_ancestor_config(path):
    for parent in [path.parent, *path.parents]:
        if parent == parent.parent:
            break
        cfg = read_json_if_exists(parent / "run_config.json")
        if cfg is not None:
            return cfg, parent / "run_config.json"
    return {}, None


def model_label_from_name(name):
    if not name:
        return None
    return str(name).split("/")[-1]


def infer_combo_and_model(path):
    parts = path.parts
    combo = None
    model = None
    family = None
    checkpoint = None
    if "results" in parts:
        family = parts[parts.index("results") + 1]

    cfg, _ = find_ancestor_config(path)
    if family and str(family).startswith("step1plus"):
        model = model_label_from_name(cfg.get("model_name_or_path")) or family.replace("step1plus_sft_section3_", "")
        combo = "gsm8k_to_mmlu"
        for candidate in ("base", "mid", "final"):
            if candidate in parts:
                checkpoint = candidate
                break
        if path.name == "all_checkpoint_section3_pairs.csv":
            checkpoint = "all"
        return family, combo, model, checkpoint

    if "one_step_ch" in parts:
        idx = parts.index("one_step_ch")
        model = parts[idx - 1]
        combo = parts[idx - 2]
        # Some older runs add an LR folder before one_step_ch.
        if re.fullmatch(r"[0-9.eE+-]+", model):
            model = parts[idx - 2]
            combo = parts[idx - 3]
    elif "section3_validation" in parts:
        idx = parts.index("section3_validation")
        model = parts[idx - 1]
        combo = parts[idx - 2]
    return family, combo, model, checkpoint


def _candidate_row(kind, path):
    family, combo, model, checkpoint = infer_combo_and_model(path)
    return {
        "kind": kind,
        "family": family,
        "combo": combo,
        "combo_key": canonical_combo(combo),
        "model": model,
        "model_key": canonical_text(model),
        "checkpoint": checkpoint,
        "checkpoint_key": canonical_text(checkpoint),
        "mtime": path.stat().st_mtime,
        "path": path,
    }


def list_available_runs(result_family=None, kind=None):
    """Return all candidate CSVs. `result_family` may be exact or a fuzzy substring."""
    candidates = []
    roots = [RESULTS_ROOT]
    exact_root = RESULTS_ROOT / str(result_family) if result_family else None
    if exact_root is not None and exact_root.exists():
        roots = [exact_root]

    for root in roots:
        if kind in (None, "one_step"):
            for pth in sorted(root.rglob("ch1_ch2_pairs.csv")):
                candidates.append(_candidate_row("one_step", pth))
        if kind in (None, "section3"):
            for pth in sorted(root.rglob("section3_pairs.csv")):
                if not str(pth).startswith(str(RESULTS_ROOT / "step1plus")) and "step1plus" not in pth.parts[pth.parts.index("results") + 1]:
                    candidates.append(_candidate_row("section3", pth))
        if kind in (None, "step1plus"):
            for pth in sorted(root.rglob("section3_pairs.csv")):
                family = pth.parts[pth.parts.index("results") + 1] if "results" in pth.parts else ""
                if str(family).startswith("step1plus"):
                    candidates.append(_candidate_row("step1plus", pth))
            for pth in sorted(root.rglob("all_checkpoint_section3_pairs.csv")):
                candidates.append(_candidate_row("step1plus", pth))

    runs = pd.DataFrame(candidates)
    if runs.empty:
        return runs

    if result_family and not (RESULTS_ROOT / str(result_family)).exists():
        fam_key = canonical_text(result_family)
        runs = runs[runs["family"].map(canonical_text).str.contains(fam_key, na=False)]
    return runs.reset_index(drop=True)


def search_runs(model=None, kind=None, combo=None, result_family=None, checkpoint=STEP1PLUS_CHECKPOINT):
    """Fuzzy search helper.

    Examples:
        search_runs("Qwen3.5-4B", "section3", "gsm8k_mmlu")
        search_runs("Qwen2.5-1.5B", "one_step", "gsm8k_dolly")
        search_runs("Qwen2.5-1.5B", "step1plus", "gsm8k_mmlu", checkpoint="final")
    """
    runs = list_available_runs(result_family=result_family, kind=kind)
    if runs.empty:
        return runs
    mask = pd.Series(True, index=runs.index)
    if model:
        model_key = canonical_text(model)
        mask &= runs["model_key"].str.contains(model_key, na=False)
    if combo:
        combo_key = canonical_combo(combo)
        mask &= runs["combo_key"].eq(combo_key)
    if kind == "step1plus" and checkpoint is not None:
        mask &= runs["checkpoint_key"].eq(canonical_text(checkpoint))
    return runs.loc[mask].sort_values(["kind", "family", "combo", "model", "checkpoint", "mtime"]).reset_index(drop=True)


def resolve_csv_path(model=MODEL, combo=DATASET_COMBO, result_family=RESULT_FAMILY, kind=SEARCH_KIND, override=CSV_PATH_OVERRIDE, checkpoint=STEP1PLUS_CHECKPOINT):
    if override is not None:
        path = Path(override)
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    candidates = search_runs(model=model, kind=kind, combo=combo, result_family=result_family, checkpoint=checkpoint)
    if len(candidates) == 1:
        return Path(candidates.iloc[0]["path"])
    if len(candidates) > 1:
        display(candidates[["kind", "family", "combo", "model", "checkpoint", "path"]])
        if AUTO_PICK_NEWEST_CANDIDATE:
            chosen = candidates.sort_values("mtime").iloc[-1]
            print("Multiple candidates matched; using newest candidate:")
            print(chosen["path"])
            return Path(chosen["path"])
        raise ValueError("Multiple matching runs. Set CSV_PATH_OVERRIDE to one exact path, or narrow RESULT_FAMILY/MODEL/DATASET_COMBO/STEP1PLUS_CHECKPOINT.")

    all_candidates = list_available_runs(result_family=result_family, kind=kind)
    print("No exact candidate matched. Nearby candidates:")
    cols = ["kind", "family", "combo", "model", "checkpoint", "path"]
    display(all_candidates.sort_values(["kind", "family", "combo", "model", "checkpoint"])[cols] if not all_candidates.empty else all_candidates)
    raise FileNotFoundError(f"Could not resolve model={model!r}, kind={kind!r}, combo={combo!r}, result_family={result_family!r}, checkpoint={checkpoint!r}")


def read_run_config(csv_path):
    for name in ("run_config.json", "config.json", "summary.json"):
        p = csv_path.parent / name
        if p.exists():
            try:
                return json.loads(p.read_text(encoding="utf-8")), p
            except Exception:
                pass
    return find_ancestor_config(csv_path)


def normalize_scatter_df(raw_df, csv_path):
    run_config, run_config_path = read_run_config(csv_path)
    lr = float(run_config.get("validation_lr", run_config.get("lr", raw_df["lr"].dropna().iloc[0] if "lr" in raw_df and raw_df["lr"].notna().any() else 1.0)))

    out = raw_df.copy()
    if "delta_logp_actual" in out:
        actual = out["delta_logp_actual"]
    elif "delta_logp" in out:
        actual = out["delta_logp"]
    else:
        raise KeyError("Could not find actual log-prob change column: expected delta_logp_actual or delta_logp")

    if "ch12_real" in out:
        first_order_raw = out["ch12_real"]
        first_order_scaled = lr * first_order_raw
    elif "first_order_exact_raw" in out:
        first_order_raw = out["first_order_exact_raw"]
        first_order_scaled = lr * first_order_raw
    elif "first_order_exact" in out:
        first_order_scaled = out["first_order_exact"]
        first_order_raw = first_order_scaled / lr
    else:
        raise KeyError("Could not find first-order column: expected ch12_real, first_order_exact_raw, or first_order_exact")

    if "ch1" in out and "ch2_approx" in out:
        approx_raw = out["ch1"] + out["ch2_approx"]
        approx_scaled = lr * approx_raw
    elif "approx_raw" in out:
        approx_raw = out["approx_raw"]
        approx_scaled = lr * approx_raw
    elif "approx" in out:
        approx_scaled = out["approx"]
        approx_raw = approx_scaled / lr
    elif "ch1" in out and "ch2" in out:
        approx_raw = out["ch1"] + out["ch2"]
        approx_scaled = lr * approx_raw
    else:
        raise KeyError("Could not find approximation columns: expected ch1+ch2_approx, approx_raw, approx, or ch1+ch2")

    norm = pd.DataFrame({
        "actual_delta_logp": pd.to_numeric(actual, errors="coerce"),
        "first_order_raw": pd.to_numeric(first_order_raw, errors="coerce"),
        "first_order_scaled": pd.to_numeric(first_order_scaled, errors="coerce"),
        "approx_raw": pd.to_numeric(approx_raw, errors="coerce"),
        "approx_scaled": pd.to_numeric(approx_scaled, errors="coerce"),
    })
    for col in ["model", "combo", "probe_domain", "observe_domain", "update_domain", "algo", "layer", "checkpoint", "adapt_step"]:
        if col in out:
            norm[col] = out[col]

    family, combo, inferred_model, checkpoint = infer_combo_and_model(csv_path)
    norm.attrs["csv_path"] = str(csv_path)
    norm.attrs["run_config_path"] = str(run_config_path) if run_config_path else None
    norm.attrs["lr"] = lr
    norm.attrs["family"] = family
    norm.attrs["combo"] = combo
    norm.attrs["model"] = inferred_model
    norm.attrs["checkpoint"] = checkpoint
    return norm

# Try this to inspect candidates interactively:
# display(search_runs(MODEL, SEARCH_KIND, DATASET_COMBO, RESULT_FAMILY, checkpoint=STEP1PLUS_CHECKPOINT))

csv_path = resolve_csv_path()
raw_df = pd.read_csv(csv_path)
df = normalize_scatter_df(raw_df, csv_path)

if SEARCH_KIND == "step1plus" and STEP1PLUS_CHECKPOINT is not None and "checkpoint" in df:
    df = df[df["checkpoint"].map(canonical_text).eq(canonical_text(STEP1PLUS_CHECKPOINT))].copy()

value_cols = ["actual_delta_logp", "first_order_scaled", "approx_scaled"]
mask = np.isfinite(df[value_cols]).all(axis=1)
if DROP_ZERO_ACTUAL:
    mask &= df["actual_delta_logp"].ne(0)
plot_df = df.loc[mask].copy()

print(f"CSV: {csv_path}")
print(f"family={df.attrs['family']} model={df.attrs['model']} combo={df.attrs['combo']} checkpoint={df.attrs.get('checkpoint')} lr={df.attrs['lr']}")
print(f"rows raw={len(raw_df):,}, selected={len(df):,}, plotted={len(plot_df):,}")
plot_df.head()


## Plot Helpers

In [ ]:
def finite_xy(x, y, positive=False):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if positive:
        mask &= (x > EPS) & (y > EPS)
    return x[mask], y[mask], mask


def corr_value(x, y, method="pearson"):
    x, y, _ = finite_xy(x, y)
    if len(x) < 2:
        return np.nan
    xs = pd.Series(x, dtype=float)
    ys = pd.Series(y, dtype=float)
    if method == "pearson":
        return float(xs.corr(ys, method="pearson"))
    if method == "spearman":
        return float(xs.rank(method="average").corr(ys.rank(method="average"), method="pearson"))
    raise ValueError(method)


def sign_agreement_value(x, y):
    x, y, _ = finite_xy(x, y)
    nonzero = (np.sign(x) != 0) & (np.sign(y) != 0)
    if nonzero.sum() == 0:
        return np.nan
    return float((np.sign(x[nonzero]) == np.sign(y[nonzero])).mean())


def corr_text(x, y, method="pearson"):
    if method in {"both", "all"}:
        p_val = corr_value(x, y, method="pearson")
        s_val = corr_value(x, y, method="spearman")

        p_str = "n < 2" if not np.isfinite(p_val) else f"{p_val:.3f}"
        s_str = "n < 2" if not np.isfinite(s_val) else f"{s_val:.3f}"
        lines = [
            f"Pearson r = {p_str}",
            rf"Spearman $\rho$ = {s_str}",
        ]
        if method == "all":
            sign_val = sign_agreement_value(x, y)
            sign_str = "n = 0" if not np.isfinite(sign_val) else f"{sign_val:.3f}"
            lines.append(f"Sign agreement = {sign_str}")
        return "\n".join(lines)

    r = corr_value(x, y, method=method)
    name = "Pearson" if method == "pearson" else "Spearman"
    stat = "r" if method == "pearson" else r"$\rho$"

    if not np.isfinite(r):
        return f"{name} {stat} = n < 2"

    return f"{name} {stat} = {r:.3f}"

def bootstrap_line_ci(x, y, x_grid, *, log_space=False, n_boot=N_BOOT, ci=CI_LEVEL, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if log_space:
        x_fit = np.log10(x)
        y_fit = np.log10(y)
        x_grid_fit = np.log10(x_grid)
    else:
        x_fit = x
        y_fit = y
        x_grid_fit = x_grid

    ok = np.isfinite(x_fit) & np.isfinite(y_fit)
    x_fit = x_fit[ok]
    y_fit = y_fit[ok]
    if len(x_fit) < 3 or np.nanstd(x_fit) == 0:
        return None, None, None

    coef = np.polyfit(x_fit, y_fit, deg=1)
    center = np.polyval(coef, x_grid_fit)

    if not SHOW_CI or n_boot <= 0:
        lower = upper = None
    else:
        preds = []
        n = len(x_fit)
        for _ in range(n_boot):
            idx = rng.integers(0, n, n)
            if np.nanstd(x_fit[idx]) == 0:
                continue
            c = np.polyfit(x_fit[idx], y_fit[idx], deg=1)
            preds.append(np.polyval(c, x_grid_fit))
        if preds:
            arr = np.vstack(preds)
            alpha = (100 - ci) / 2
            lower = np.percentile(arr, alpha, axis=0)
            upper = np.percentile(arr, 100 - alpha, axis=0)
        else:
            lower = upper = None

    if log_space:
        center = 10 ** center
        if lower is not None:
            lower = 10 ** lower
            upper = 10 ** upper
    return center, lower, upper




def scatter_with_fit(
    ax, x, y, *,
    xlabel, ylabel, title=None,
    corr_method="pearson",
    loglog=False,
    symlog=False,
    rank=False,
):
    x, y, _ = finite_xy(x, y, positive=loglog)

    ax.scatter(
        x, y,
        marker=MARKER,
        s=SCATTER_SIZE,
        alpha=SCATTER_ALPHA,
        color=COLORS["scatter"],
    )

    if loglog:
        ax.set_xscale("log")
        ax.set_yscale("log")
    elif symlog:
        linthresh_x = max(1e-9, np.percentile(np.abs(x), 10))
        linthresh_y = max(1e-9, np.percentile(np.abs(y), 10))
        ax.set_xscale("symlog", linthresh=linthresh_x)
        ax.set_yscale("symlog", linthresh=linthresh_y)
        ticks = [-1e-2, -1e-4, -1e-6, 0, 1e-6, 1e-4, 1e-2]
        
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)

    if SHOW_FIT_LINE and len(x) >= 3 and not symlog:
        if loglog:
            x_grid = np.geomspace(x.min(), x.max(), 200)
            center, lower, upper = bootstrap_line_ci(x, y, x_grid, log_space=True)
        else:
            x_grid = np.linspace(x.min(), x.max(), 200)
            center, lower, upper = bootstrap_line_ci(x, y, x_grid, log_space=False)

        if center is not None:
            ax.plot(x_grid, center, color=COLORS["fit"], lw=1.5)
            if lower is not None and upper is not None:
                ax.fill_between(
                    x_grid, lower, upper,
                    color=COLORS["ci"], alpha=0.18, linewidth=0
                )

    text = corr_text(x, y, method=corr_method)
    ax.text(
        0.04, 0.96, text,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25"},
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.grid(True, alpha=0.22)
    return ax


def save_figure(fig, stem):
    saved = []
    if SAVE_PNG:
        p = FIGURE_DIR / f"{stem}.png"
        fig.savefig(p, dpi=300, bbox_inches="tight")
        saved.append(p)
    if SAVE_PDF:
        p = FIGURE_DIR / f"{stem}.pdf"
        fig.savefig(p, dpi=150, bbox_inches="tight")
        saved.append(p)
    print("saved:", *saved, sep="\n  ")
    return saved


def figure_stem(prefix):
    model = str(df.attrs.get("model", MODEL)).replace("/", "_")
    combo = str(df.attrs.get("combo", DATASET_COMBO))
    return f"{prefix}_{combo}_{model}"

## Figure A: Actual, First-Order, and CH1+CH2 Approximation

In [ ]:
def plot_single_panel(x, y, *, xlabel, ylabel, title, corr_method, loglog, stem):
    fig, ax = plt.subplots(1, 1, figsize=(4.2, 3.7))
    scatter_with_fit(
        ax, x, y,
        xlabel=xlabel,
        ylabel=ylabel,
        title=title,
        corr_method=corr_method,
        loglog=loglog,
    )
    if not loglog:
        ax.axhline(0, color="0.75", lw=0.8)
        ax.axvline(0, color="0.75", lw=0.8)
    fig.tight_layout()
    save_figure(fig, figure_stem(stem))
    return fig, ax


def plot_figure_a_separate(data=plot_df):
    actual = data["actual_delta_logp"]
    first = data["first_order_scaled"]
    approx = data["approx_scaled"]

    figs = []
    figs.append(plot_single_panel(
        first, actual,
        xlabel=LABELS["first_order"],
        ylabel=LABELS["actual"],
        #title="Actual vs first-order",
        title=None,
        corr_method="both",
        loglog=False,
        stem="figureA1_actual_vs_first_order",
    ))
    figs.append(plot_single_panel(
        #np.abs(first), np.abs(approx),
        first, approx,
        xlabel=LABELS["first_order"],
        ylabel=LABELS["approx"],
        #title="Magnitude: first-order vs CH1+CH2",
        title=None,
        corr_method="both",
        loglog=False,
        stem="figureA2_abs_first_order_vs_abs_approx",
    ))
    figs.append(plot_single_panel(
        #np.abs(actual), np.abs(approx),
        actual, approx,
        xlabel=LABELS["actual"],
        ylabel=LABELS["approx"],
        title=None,#"Magnitude: actual vs CH1+CH2",
        corr_method="both",
        loglog=False,
        stem="figureA3_abs_actual_vs_abs_approx",
    ))
    return figs

plot_figure_a_separate();


## Figure B: Absolute Magnitude Scatter and Rank Scatter

In [ ]:
def plot_figure_b(data=plot_df):
    first_abs = np.abs(data["first_order_scaled"])
    approx_abs = np.abs(data["approx_scaled"])
    first_rank = first_abs.rank(method="average")
    approx_rank = approx_abs.rank(method="average")

    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.6))

    scatter_with_fit(
        axes[0], first_abs, approx_abs,
        xlabel=LABELS["abs_first_order"],
        ylabel=LABELS["abs_approx"],
        title="Absolute magnitude",
        corr_method="both",
        loglog=True,
    )

    scatter_with_fit(
        axes[1], first_rank, approx_rank,
        xlabel=LABELS["rank_abs_first_order"],
        ylabel=LABELS["rank_abs_approx"],
        title="Absolute rank",
        corr_method="both",
        loglog=False,
    )

    for label, ax in zip(["a1", "a2"], axes):
        ax.text(-0.16, 1.06, label, transform=ax.transAxes, fontweight="bold", fontsize=12)

    fig.tight_layout()
    save_figure(fig, figure_stem("figureB_abs_rank_ch"))
    return fig, axes

plot_figure_b();

## Figure C: Signed Scatter and Signed Rank Scatter

In [ ]:
def plot_figure_c(data=plot_df):
    first = data["first_order_scaled"]
    approx = data["approx_scaled"]
    first_rank = first.rank(method="average")
    approx_rank = approx.rank(method="average")

    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.6))

    scatter_with_fit(
        axes[0], first, approx,
        xlabel=LABELS["first_order"],
        ylabel=LABELS["approx"],
        title="Signed values",
        corr_method="both",
        loglog=False,
        symlog=True
    )
    axes[0].axhline(0, color="0.75", lw=0.8)
    axes[0].axvline(0, color="0.75", lw=0.8)

    scatter_with_fit(
        axes[1], first_rank, approx_rank,
        xlabel=LABELS["rank_first_order"],
        ylabel=LABELS["rank_approx"],
        title="Signed rank",
        corr_method="both",
        loglog=False,
    )

    for label, ax in zip(["b1", "b2"], axes):
        ax.text(-0.16, 1.06, label, transform=ax.transAxes, fontweight="bold", fontsize=12)

    fig.tight_layout()
    save_figure(fig, figure_stem("figureC_signed_rank_ch"))
    return fig, axes

plot_figure_c();

## Inspect Available Runs and Data

In [ ]:
# Useful when switching MODEL / DATASET_COMBO above.
available_runs = list_available_runs(RESULT_FAMILY, SEARCH_KIND).sort_values(["combo", "model", "kind", "family"])
display(available_runs[["kind", "family", "combo", "model", "checkpoint", "path"]])

print("Current search candidates:")
display(search_runs(MODEL, SEARCH_KIND, DATASET_COMBO, RESULT_FAMILY, checkpoint=STEP1PLUS_CHECKPOINT)[["kind", "family", "combo", "model", "checkpoint", "path"]])

print("Normalized columns:")
display(plot_df.describe())
print("Raw columns:")
print(list(raw_df.columns))
